In [2]:
import pandas as pd
import numpy as np
import gc
import matplotlib.pyplot as plt
import seaborn as sns

print("UCITAVANJE PODATAKA")

df = pd.read_csv('../../archive/yellow_tripdata_2015-01.csv')
print(f"Ucitano: {len(df):,} redova, {len(df.columns)} kolona")

UCITAVANJE PODATAKA
Ucitano: 12,748,986 redova, 19 kolona


In [3]:
print("PROVERA NEDOSTAJUCIH VREDNOSTI")

missing_values = df.isnull().sum()
missing_cols = missing_values[missing_values > 0]
if len(missing_cols) > 0:
    print(missing_cols)
else:
    print("Nema nedostajucih vrednosti!")

PROVERA NEDOSTAJUCIH VREDNOSTI
improvement_surcharge    3
dtype: int64


In [4]:
print("\n" + "="*60)
print("PROVERA EKSTREMNIH VREDNOSTI (OUTLIER-A)")
print("="*60)

zero_distance = df[df['trip_distance'] == 0]
print(f"Vožnje sa 0 milja: {len(zero_distance):,}")

long_distance = df[df['trip_distance'] > 100]
print(f"Vožnje preko 100 milja: {len(long_distance):,}")

zero_fare = df[df['fare_amount'] <= 0]
print(f"Vožnje sa iznosom <= 0: {len(zero_fare):,}")

high_fare = df[df['fare_amount'] > 500]
print(f"Vožnje sa iznosom > 500$: {len(high_fare):,}")

invalid_passengers = df[(df['passenger_count'] == 0) | (df['passenger_count'] > 6)]
print(f"Vožnje sa neispravnim brojem putnika (0 ili >6): {len(invalid_passengers):,}")

negative_tip = df[df['tip_amount'] < 0]
print(f"Vožnje sa negativnom napojnicom: {len(negative_tip):,}")


PROVERA EKSTREMNIH VREDNOSTI (OUTLIER-A)
Vožnje sa 0 milja: 79,365
Vožnje preko 100 milja: 172
Vožnje sa iznosom <= 0: 7,666
Vožnje sa iznosom > 500$: 77
Vožnje sa neispravnim brojem putnika (0 ili >6): 6,595
Vožnje sa negativnom napojnicom: 79


In [5]:
print("PROVERA VREMENSKIH NEKONZISTENCIJA")

# Konverzija vremena
df['tpep_pickup_datetime'] = pd.to_datetime(df['tpep_pickup_datetime'])
df['tpep_dropoff_datetime'] = pd.to_datetime(df['tpep_dropoff_datetime'])

# Provera dropoff pre pickup
invalid_time = df[df['tpep_dropoff_datetime'] < df['tpep_pickup_datetime']]
print(f"Vožnje sa dropoff pre pickup: {len(invalid_time):,}")

# Trajanje vožnje
df['duration_minutes'] = (df['tpep_dropoff_datetime'] - df['tpep_pickup_datetime']).dt.total_seconds() / 60

long_rides = df[df['duration_minutes'] > 1440]  # 24h = 1440 min
print(f"Vožnje duže od 24h: {len(long_rides):,}")

zero_duration = df[df['duration_minutes'] == 0]
print(f"Vožnje koje traju 0 minuta: {len(zero_duration):,}")

PROVERA VREMENSKIH NEKONZISTENCIJA
Vožnje sa dropoff pre pickup: 297
Vožnje duže od 24h: 79
Vožnje koje traju 0 minuta: 14,816


In [6]:
print("OSNOVNO CISCENJE")
print(f"Pre ciscenja: {len(df):,}")

df_clean = df[
    # Vreme
    (df['tpep_dropoff_datetime'] >= df['tpep_pickup_datetime']) &
    (df['duration_minutes'] > 0) &
    (df['duration_minutes'] <= 1440) &
    
    # Distanca
    (df['trip_distance'] > 0) &
    (df['trip_distance'] <= 100) &
    
    # Cena
    (df['fare_amount'] > 0) &
    (df['fare_amount'] <= 500) &
    (df['tip_amount'] >= 0) &
    
    # Putnici
    (df['passenger_count'] >= 1) &
    (df['passenger_count'] <= 6)
].copy()

print(f"Posle osnovnog ciscenja: {len(df_clean):,}")
print(f"Uklonjeno: {len(df) - len(df_clean):,} redova ({((len(df)-len(df_clean))/len(df)*100):.2f}%)")

# Oslobadjam memoriju
del df
gc.collect()

OSNOVNO CISCENJE
Pre ciscenja: 12,748,986
Posle osnovnog ciscenja: 12,657,281
Uklonjeno: 91,705 redova (0.72%)


1123

In [7]:
print("POPUNJAVANJE NULL VREDNOSTI")
print(f"NULL pre popunjavanja:")
null_counts = df_clean.isnull().sum()
print(null_counts[null_counts > 0])

# Popuni NULL vrednosti
df_clean['improvement_surcharge'] = df_clean['improvement_surcharge'].fillna(0)
df_clean['store_and_fwd_flag'] = df_clean['store_and_fwd_flag'].fillna('N')

print(f"NULL posle popunjavanja: {df_clean.isnull().sum().sum()}")

POPUNJAVANJE NULL VREDNOSTI
NULL pre popunjavanja:
improvement_surcharge    3
dtype: int64
NULL posle popunjavanja: 0


In [8]:
print("FILTRIRANJE KOORDINATA (NYC OPSEG)")
before = len(df_clean)

df_clean = df_clean[
    (df_clean['pickup_longitude'].between(-74.5, -73.5)) &
    (df_clean['pickup_latitude'].between(40.5, 41.0)) &
    (df_clean['dropoff_longitude'].between(-74.5, -73.5)) &
    (df_clean['dropoff_latitude'].between(40.5, 41.0))
]

print(f"Uklonjeno van NYC opsega: {before - len(df_clean):,} redova")

FILTRIRANJE KOORDINATA (NYC OPSEG)
Uklonjeno van NYC opsega: 234,683 redova


In [9]:
print("UKLANJANJE OUTLIER-A (IQR METODA)")
def remove_outliers_iqr(df, column, multiplier=1.5):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - multiplier * IQR
    upper = Q3 + multiplier * IQR
    
    before = len(df)
    df_clean = df[(df[column] >= lower) & (df[column] <= upper)]
    
    print(f"  {column}:")
    print(f"    Opseg: {lower:.2f} - {upper:.2f}")
    print(f"    Uklonjeno: {before - len(df_clean):,} redova")
    
    return df_clean

outlier_cols = ['trip_distance', 'fare_amount', 'total_amount']
for col in outlier_cols:
    df_clean = remove_outliers_iqr(df_clean, col)

UKLANJANJE OUTLIER-A (IQR METODA)
  trip_distance:
    Opseg: -2.01 - 6.02
    Uklonjeno: 1,231,005 redova
  fare_amount:
    Opseg: -2.25 - 19.75
    Uklonjeno: 277,301 redova
  total_amount:
    Opseg: -1.20 - 22.80
    Uklonjeno: 177,110 redova


In [10]:
print("FILTRIRANJE KATEGORIJALNIH VARIJABLI")
before = len(df_clean)

df_clean = df_clean[df_clean['VendorID'].isin([1, 2])]
df_clean = df_clean[df_clean['RateCodeID'].isin([1, 2, 3, 4, 5, 6])]
df_clean = df_clean[df_clean['payment_type'].isin([1, 2, 3, 4, 5, 6])]

print(f"Uklonjeno sa nevalidnim kodovima: {before - len(df_clean):,}")

FILTRIRANJE KATEGORIJALNIH VARIJABLI
Uklonjeno sa nevalidnim kodovima: 195


In [11]:
print("UKLANJANJE DUPLIKATA")
before = len(df_clean)

# Samo potpuni duplikati (isti pickup, dropoff, vreme)
df_clean = df_clean.drop_duplicates(
    subset=['pickup_longitude', 'pickup_latitude', 
            'dropoff_longitude', 'dropoff_latitude',
            'tpep_pickup_datetime', 'tpep_dropoff_datetime']
)

print(f"Uklonjeno potpunih duplikata: {before - len(df_clean):,}")


UKLANJANJE DUPLIKATA
Uklonjeno potpunih duplikata: 0


In [12]:
# Brzina (milja po satu)
df_clean['avg_speed_mph'] = df_clean['trip_distance'] / (df_clean['duration_minutes'] / 60)

# Cena po milji
df_clean['price_per_mile'] = df_clean['fare_amount'] / df_clean['trip_distance']

# Vremenske karakteristike
df_clean['pickup_hour'] = df_clean['tpep_pickup_datetime'].dt.hour
df_clean['pickup_day'] = df_clean['tpep_pickup_datetime'].dt.dayofweek
df_clean['pickup_month'] = df_clean['tpep_pickup_datetime'].dt.month
df_clean['pickup_day_name'] = df_clean['tpep_pickup_datetime'].dt.day_name()
df_clean['is_weekend'] = (df_clean['pickup_day'] >= 5).astype(int)

# Rush hour  (7-9 i 16-19)
df_clean['is_rush_hour'] = df_clean['pickup_hour'].isin([7, 8, 9, 16, 17, 18, 19]).astype(int)

print(f"Dodato 7 novih karakteristika")
print(f"Ukupno kolona: {len(df_clean.columns)}")


Dodato 7 novih karakteristika
Ukupno kolona: 28


In [14]:
print("KONAČAN BROJ REDOVA I KOLONA")
print(f"Broj redova: {len(df_clean):,}")
print(f"Broj kolona: {len(df_clean.columns)}")

KONAČAN BROJ REDOVA I KOLONA
Broj redova: 10,736,987
Broj kolona: 28


In [15]:
print("CUVANJE OCISCENIH PODATAKA")
putanja = r'C:\Users\Andjela\Desktop\yellow_tripdata_2015-01_OCISCEN.csv'
df_clean.to_csv(putanja, index=False)
print(f"Fajl sacuvan na: {putanja}")
print("CISCENJE ZAVRSENO!")

CUVANJE OCISCENIH PODATAKA
Fajl sacuvan na: C:\Users\Andjela\Desktop\yellow_tripdata_2015-01_OCISCEN.csv
CISCENJE ZAVRSENO!


In [1]:
import pandas as pd
from sqlalchemy import create_engine
import pyodbc

server = 'DESKTOP-64J7LMV\\MSSQLSERVER2022' 
database = 'PROJEKAT'           

engine = create_engine(
    f'mssql+pyodbc://{server}/{database}?driver=ODBC+Driver+17+for+SQL+Server&Trusted_Connection=yes'
)

try:
    with engine.connect() as conn:
        print("Povezano!")
except Exception as e:
    print(f"Greška! {e}")



Povezano!


In [3]:
# DIM_DATE
print("KREIRANJE DIM_DATE")

df_clean = pd.read_csv('../../archive/yellow_tripdata_2015-01_OCISCEN.csv')

df_clean['tpep_pickup_datetime'] = pd.to_datetime(
    df_clean['tpep_pickup_datetime']
)
print(f"Učitano: {len(df_clean):,} redova")
print(f"Broj kolona: {len(df_clean.columns)}")

dim_date = df_clean[['tpep_pickup_datetime']].copy()
dim_date['Date_ID'] = dim_date['tpep_pickup_datetime'].dt.strftime('%Y%m%d').astype(int)
dim_date['Full_Date'] = dim_date['tpep_pickup_datetime'].dt.date
dim_date['Year'] = dim_date['tpep_pickup_datetime'].dt.year
dim_date['Month'] = dim_date['tpep_pickup_datetime'].dt.month
dim_date['Month_Name'] = dim_date['tpep_pickup_datetime'].dt.strftime('%B')
dim_date['Day'] = dim_date['tpep_pickup_datetime'].dt.day
dim_date['Day_Of_Week'] = dim_date['tpep_pickup_datetime'].dt.dayofweek + 1
dim_date['Day_Name'] = dim_date['tpep_pickup_datetime'].dt.strftime('%A')
dim_date['Is_Weekend'] = (dim_date['Day_Of_Week'] >= 6).astype(int)

# Izbaci duplikate
dim_date = dim_date.drop_duplicates(subset=['Date_ID'])

# Odabir kolone za SQL
dim_date_to_sql = dim_date[['Date_ID', 'Full_Date', 'Year', 'Month', 'Month_Name', 
                            'Day', 'Day_Of_Week', 'Day_Name', 'Is_Weekend']]

print(f"Broj redova: {len(dim_date_to_sql):,}")
print(dim_date_to_sql.head())

# Učitaj u bazu
dim_date_to_sql.to_sql('DIM_DATE', engine, if_exists='replace', index=False)
print("DIM_DATE uspešno učitana!")

KREIRANJE DIM_DATE
Učitano: 10,736,987 redova
Broj kolona: 28
Broj redova: 31
     Date_ID   Full_Date  Year  Month Month_Name  Day  Day_Of_Week  Day_Name  \
0   20150115  2015-01-15  2015      1    January   15            4  Thursday   
1   20150110  2015-01-10  2015      1    January   10            6  Saturday   
29  20150125  2015-01-25  2015      1    January   25            7    Sunday   
47  20150104  2015-01-04  2015      1    January    4            7    Sunday   
75  20150126  2015-01-26  2015      1    January   26            1    Monday   

    Is_Weekend  
0            0  
1            1  
29           1  
47           1  
75           0  
DIM_DATE uspešno učitana!


In [4]:
print("KREIRANJE DIM_TIME")
# Kreiranje dim_time
dim_time = df_clean[['tpep_pickup_datetime']].copy()
dim_time['Time_ID'] = dim_time['tpep_pickup_datetime'].dt.hour * 100 + dim_time['tpep_pickup_datetime'].dt.minute
dim_time['Hour'] = dim_time['tpep_pickup_datetime'].dt.hour
dim_time['Minute'] = dim_time['tpep_pickup_datetime'].dt.minute

def get_time_of_day(hour):
    if 5 <= hour < 12:
        return 'Jutro'
    elif 12 <= hour < 17:
        return 'Podne'
    elif 17 <= hour < 21:
        return 'Veče'
    else:
        return 'Noć'

dim_time['Time_Of_Day'] = dim_time['Hour'].apply(get_time_of_day)

# Izbaci duplikate
dim_time = dim_time.drop_duplicates(subset=['Time_ID'])

# Odabir kolone za SQL
dim_time_to_sql = dim_time[['Time_ID', 'Hour', 'Minute', 'Time_Of_Day']]

print(f"Broj redova: {len(dim_time_to_sql):,}")
print(dim_time_to_sql.head())

# Učitaj u bazu
dim_time_to_sql.to_sql('DIM_TIME', engine, if_exists='replace', index=False)
print("DIM_TIME uspešno učitana!")

KREIRANJE DIM_TIME
Broj redova: 1,440
    Time_ID  Hour  Minute Time_Of_Day
0      1905    19       5        Veče
1      2033    20      33        Veče
28     1912    19      12        Veče
29       13     0      13         Noć
47     1344    13      44       Podne
DIM_TIME uspešno učitana!


In [5]:
import pandas as pd
import numpy as np
df_clean = pd.read_csv('../../archive/yellow_tripdata_2015-01_OCISCEN.csv')
print(f"Učitano: {len(df_clean):,} redova")
print(f"Broj kolona: {len(df_clean.columns)}")

def get_zone(lat, lon):
    """Vraća ime zone na osnovu koordinata"""
    if pd.isna(lat) or pd.isna(lon):
        return 'Unknown'
    
    # Manhattan
    if 40.70 <= lat <= 40.88 and -74.02 <= lon <= -73.91:
        return 'Manhattan'
    # Brooklyn
    elif 40.57 <= lat <= 40.74 and -74.06 <= lon <= -73.83:
        return 'Brooklyn'
    # Queens
    elif 40.55 <= lat <= 40.80 and -73.96 <= lon <= -73.70:
        return 'Queens'
    # Bronx
    elif 40.80 <= lat <= 40.92 and -73.93 <= lon <= -73.76:
        return 'Bronx'
    # Staten Island
    elif 40.50 <= lat <= 40.65 and -74.26 <= lon <= -74.05:
        return 'Staten Island'
    # JFK Airport
    elif 40.63 <= lat <= 40.66 and -73.80 <= lon <= -73.75:
        return 'JFK Airport'
    # Newark Airport
    elif 40.68 <= lat <= 40.72 and -74.19 <= lon <= -74.15:
        return 'Newark Airport'
    # LaGuardia Airport
    elif 40.76 <= lat <= 40.78 and -73.89 <= lon <= -73.85:
        return 'LaGuardia Airport'
    else:
        return 'Other'
print("KREIRANJE DIM_PICKUP_LOCATION")

print("Dodajem zone za pickup lokacije...")
df_clean['pickup_zone'] = df_clean.apply(
    lambda row: get_zone(row['pickup_latitude'], row['pickup_longitude']), 
    axis=1
)
print("Zone dodate!")

dim_pickup = df_clean[['pickup_longitude', 'pickup_latitude', 'pickup_zone']].rename(
    columns={
        'pickup_longitude': 'Longitude', 
        'pickup_latitude': 'Latitude',
        'pickup_zone': 'Zone'
    }
).drop_duplicates().dropna()
dim_pickup = dim_pickup.reset_index(drop=True)
dim_pickup.index = dim_pickup.index + 1
dim_pickup['Pickup_Location_ID'] = dim_pickup.index

dim_pickup = dim_pickup[['Pickup_Location_ID', 'Longitude', 'Latitude', 'Zone']]

print(f"Broj redova: {len(dim_pickup):,}")

print("\nDistribucija zona (pickup):")
print(dim_pickup['Zone'].value_counts())

print("KREIRANJE DIM_DROPOFF_LOCATION")

print("Dodajem zone za dropoff lokacije...")
df_clean['dropoff_zone'] = df_clean.apply(
    lambda row: get_zone(row['dropoff_latitude'], row['dropoff_longitude']), 
    axis=1
)

dim_dropoff = df_clean[['dropoff_longitude', 'dropoff_latitude', 'dropoff_zone']].rename(
    columns={
        'dropoff_longitude': 'Longitude', 
        'dropoff_latitude': 'Latitude',
        'dropoff_zone': 'Zone'
    }
).drop_duplicates().dropna()

dim_dropoff = dim_dropoff.reset_index(drop=True)
dim_dropoff.index = dim_dropoff.index + 1
dim_dropoff['Dropoff_Location_ID'] = dim_dropoff.index

dim_dropoff = dim_dropoff[['Dropoff_Location_ID', 'Longitude', 'Latitude', 'Zone']]

print(f"Broj redova: {len(dim_dropoff):,}")
print("\nDistribucija zona (dropoff):")
print(dim_dropoff['Zone'].value_counts())

print("REZIME")
print(f"DIM_PICKUP_LOCATION:   {len(dim_pickup):,} redova")
print(f"DIM_DROPOFF_LOCATION:  {len(dim_dropoff):,} redova")
print("\n Dimenzije lokacija uspešno kreirane!")

Učitano: 10,736,987 redova
Broj kolona: 28
KREIRANJE DIM_PICKUP_LOCATION
Dodajem zone za pickup lokacije...
Zone dodate!
Broj redova: 8,548,109

Distribucija zona (pickup):
Zone
Manhattan         8388714
Brooklyn           107765
Queens              48511
Bronx                2568
Other                 462
Staten Island          56
Newark Airport         33
Name: count, dtype: int64
KREIRANJE DIM_DROPOFF_LOCATION
Dodajem zone za dropoff lokacije...
Broj redova: 9,102,212

Distribucija zona (dropoff):
Zone
Manhattan         8853644
Brooklyn           188208
Queens              54440
Bronx                5305
Other                 525
Staten Island          56
Newark Airport         34
Name: count, dtype: int64
REZIME
DIM_PICKUP_LOCATION:   8,548,109 redova
DIM_DROPOFF_LOCATION:  9,102,212 redova

 Dimenzije lokacija uspešno kreirane!


In [8]:
dim_pickup.to_sql('DIM_PICKUP_LOCATION', engine, if_exists='replace', index=False)
print("DIM_PICKUP_LOCATION učitana!")

# Učitaj dim_dropoff u bazu
dim_dropoff.to_sql('DIM_DROPOFF_LOCATION', engine, if_exists='replace', index=False)
print("DIM_DROPOFF_LOCATION učitana!")

DIM_PICKUP_LOCATION učitana!
DIM_DROPOFF_LOCATION učitana!


In [14]:
print("KREIRANJE DIM_VENDOR")

dim_vendor = pd.DataFrame([
    [1, 'Creative Mobile Technologies'],
    [2, 'VeriFone Inc.']
], columns=['Vendor_ID', 'Vendor_Name'])  

print(dim_vendor)
dim_vendor.to_sql('DIM_VENDOR', engine, if_exists='replace', index=False)
print(" DIM_VENDOR učitana!")

KREIRANJE DIM_VENDOR
   Vendor_ID                   Vendor_Name
0          1  Creative Mobile Technologies
1          2                 VeriFone Inc.
 DIM_VENDOR učitana!


In [15]:
print("KREIRANJE DIM_RATE_CODE")
dim_rate_code = pd.DataFrame([
    [1, 'Standard rate'],
    [2, 'JFK'],
    [3, 'Newark'],
    [4, 'Nassau or Westchester'],
    [5, 'Negotiated fare'],
    [6, 'Group ride']
], columns=['RateCode_ID', 'RateCode_Name'])

print(f"Broj redova: {len(dim_rate_code)}")
print(dim_rate_code)
dim_rate_code.to_sql('DIM_RATE_CODE', engine, if_exists='replace', index=False)

KREIRANJE DIM_RATE_CODE
Broj redova: 6
   RateCode_ID          RateCode_Name
0            1          Standard rate
1            2                    JFK
2            3                 Newark
3            4  Nassau or Westchester
4            5        Negotiated fare
5            6             Group ride


6

In [16]:
print("KREIRANJE DIM_PAYMENT_TYPE")
dim_payment_type = pd.DataFrame([
    [1, 'Credit card'],
    [2, 'Cash'],
    [3, 'No charge'],
    [4, 'Dispute'],
    [5, 'Unknown'],
    [6, 'Voided trip']
], columns=['Payment_ID', 'Payment_Name'])

print(f"Broj redova: {len(dim_payment_type)}")
print(dim_payment_type)
dim_payment_type.to_sql('DIM_PAYMENT_TYPE', engine, if_exists='replace', index=False)

KREIRANJE DIM_PAYMENT_TYPE
Broj redova: 6
   Payment_ID Payment_Name
0           1  Credit card
1           2         Cash
2           3    No charge
3           4      Dispute
4           5      Unknown
5           6  Voided trip


6

In [9]:
# ============================================
# FACT_TRIP - ČISTI SQL (BEZ PANDAS MAPIRANJA)
# ============================================
import gc
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text

print("="*60)
print("FACT_TRIP - ČISTI SQL")
print("="*60)

# 1. KONEKCIJA NA BAZU
server = 'DESKTOP-64J7LMV\\MSSQLSERVER2022' 
database = 'PROJEKAT'           

engine = create_engine(
    f'mssql+pyodbc://{server}/{database}?driver=ODBC+Driver+17+for+SQL+Server&Trusted_Connection=yes'
)

try:
    with engine.connect() as conn:
        print(" Povezano!")
except Exception as e:
    print(f" Greška! {e}")

# 2. OBRISI STARU FACT_TRIP
try:
    with engine.connect() as conn:
        conn.execute(text("DROP TABLE IF EXISTS FACT_TRIP"))
        conn.commit()
    print("Stara FACT_TRIP obrisana")
except:
    pass

# 3. KREIRANJE FACT_TRIP TABELE U SQL
print("\nKreiram FACT_TRIP tabelu u SQL...")

create_table_sql = """
CREATE TABLE FACT_TRIP (
    Date_ID INT,
    Time_ID INT,
    Pickup_Location_ID INT,
    Dropoff_Location_ID INT,
    Vendor_ID INT,
    RateCode_ID INT,
    Payment_ID INT,
    passenger_count INT,
    trip_distance FLOAT,
    fare_amount FLOAT,
    extra FLOAT,
    mta_tax FLOAT,
    tip_amount FLOAT,
    improvement_surcharge FLOAT,
    total_amount FLOAT,
    trip_duration FLOAT
)
"""

try:
    with engine.connect() as conn:
        conn.execute(text(create_table_sql))
        conn.commit()
    print(" FACT_TRIP tabela kreirana!")
except Exception as e:
    print(f"Tabela verovatno već postoji: {e}")
print("\nUčitavam podatke u FACT_TRIP...")

chunk_size = 50000
total_rows = 0

for chunk in pd.read_csv(
    '../../archive/yellow_tripdata_2015-01_OCISCEN.csv',
    chunksize=chunk_size,
    usecols=['VendorID', 'tpep_pickup_datetime', 'pickup_longitude', 'pickup_latitude',
             'dropoff_longitude', 'dropoff_latitude', 'RateCodeID', 'payment_type',
             'passenger_count', 'trip_distance', 'fare_amount', 'extra', 'mta_tax',
             'tip_amount', 'improvement_surcharge', 'total_amount', 'duration_minutes']
):
    # Konvertuj datetime
    chunk['tpep_pickup_datetime'] = pd.to_datetime(chunk['tpep_pickup_datetime'])
    
    # Dodaj Date_ID i Time_ID
    chunk['Date_ID'] = chunk['tpep_pickup_datetime'].dt.strftime('%Y%m%d').astype(int)
    chunk['Time_ID'] = chunk['tpep_pickup_datetime'].dt.hour * 100 + chunk['tpep_pickup_datetime'].dt.minute
    
    # Preimenuj duration_minutes
    chunk = chunk.rename(columns={'duration_minutes': 'trip_duration'})
    
    # Učitaj chunk u temporary tabelu
    chunk.to_sql('#temp_chunk', engine, if_exists='replace', index=False)
    
    insert_sql = """
    INSERT INTO FACT_TRIP (
        Date_ID, Time_ID, Pickup_Location_ID, Dropoff_Location_ID,
        Vendor_ID, RateCode_ID, Payment_ID,
        passenger_count, trip_distance, fare_amount,
        extra, mta_tax, tip_amount,
        improvement_surcharge, total_amount, trip_duration
    )
    SELECT 
        t.Date_ID,
        t.Time_ID,
        p.Pickup_Location_ID,
        d.Dropoff_Location_ID,
        v.Vendor_ID,
        r.RateCode_ID,
        pay.Payment_ID,
        t.passenger_count,
        t.trip_distance,
        t.fare_amount,
        t.extra,
        t.mta_tax,
        t.tip_amount,
        t.improvement_surcharge,
        t.total_amount,
        t.trip_duration
    FROM #temp_chunk t
    LEFT JOIN DIM_PICKUP_LOCATION p 
        ON t.pickup_longitude = p.Longitude 
        AND t.pickup_latitude = p.Latitude
    LEFT JOIN DIM_DROPOFF_LOCATION d 
        ON t.dropoff_longitude = d.Longitude 
        AND t.dropoff_latitude = d.Latitude
    LEFT JOIN DIM_VENDOR v ON t.VendorID = v.Vendor_ID
    LEFT JOIN DIM_RATE_CODE r ON t.RateCodeID = r.RateCode_ID
    LEFT JOIN DIM_PAYMENT_TYPE pay ON t.payment_type = pay.Payment_ID
    WHERE 
        p.Pickup_Location_ID IS NOT NULL
        AND d.Dropoff_Location_ID IS NOT NULL
        AND v.Vendor_ID IS NOT NULL
        AND r.RateCode_ID IS NOT NULL
        AND pay.Payment_ID IS NOT NULL
    """
    
    try:
        with engine.connect() as conn:
            conn.execute(text(insert_sql))
            conn.commit()
    except Exception as e:
        print(f"Greška pri INSERT-u: {e}")
        # Obriši temp tabelu i nastavi
        with engine.connect() as conn:
            conn.execute(text("DROP TABLE IF EXISTS #temp_chunk"))
            conn.commit()
        continue
    
    # Obriši temporary tabelu
    with engine.connect() as conn:
        conn.execute(text("DROP TABLE IF EXISTS #temp_chunk"))
        conn.commit()
    
    total_rows += chunk_size
    print(f"  Učitano: {total_rows:,} redova", end="\r")
    
    # Oslobodi memoriju
    del chunk
    gc.collect()

print(f"\nFACT_TRIP učitana!")

print("PROVERA")

result = pd.read_sql("SELECT COUNT(*) AS broj FROM FACT_TRIP", engine)
print(f"FACT_TRIP u bazi: {result['broj'].iloc[0]:,} redova")

# Prikaži prvih 5 redova
sample = pd.read_sql("SELECT TOP 5 * FROM FACT_TRIP", engine)
print("\nPrvih 5 redova:")
print(sample)

FACT_TRIP - ČISTI SQL
✅ Povezano!
Stara FACT_TRIP obrisana

Kreiram FACT_TRIP tabelu u SQL...
✅ FACT_TRIP tabela kreirana!

Učitavam podatke u FACT_TRIP...
  Učitano: 10,750,000 redova
✅ FACT_TRIP učitana!

PROVERA
FACT_TRIP u bazi: 10,736,987 redova

Prvih 5 redova:
    Date_ID  Time_ID  Pickup_Location_ID  Dropoff_Location_ID  Vendor_ID  \
0  20150113     1150               22361                22364          1   
1  20150113     1150               22362                22365          1   
2  20150113     1150               22363                22366          1   
3  20150113     1150               22364                22367          1   
4  20150113     1150               22365                22368          1   

   RateCode_ID  Payment_ID  passenger_count  trip_distance  fare_amount  \
0            1           2                1            0.8          5.0   
1            1           2                1            0.6          6.5   
2            1           2                1       

In [1]:
import pandas as pd

df = pd.read_csv('../../archive/yellow_tripdata_2015-01_OCISCEN.csv',nrows=1)

for col in df.columns:
    print(f"{col:<30} -> {df[col].dtype}")

VendorID                       -> int64
tpep_pickup_datetime           -> str
tpep_dropoff_datetime          -> str
passenger_count                -> int64
trip_distance                  -> float64
pickup_longitude               -> float64
pickup_latitude                -> float64
RateCodeID                     -> int64
store_and_fwd_flag             -> str
dropoff_longitude              -> float64
dropoff_latitude               -> float64
payment_type                   -> int64
fare_amount                    -> float64
extra                          -> float64
mta_tax                        -> float64
tip_amount                     -> float64
tolls_amount                   -> float64
improvement_surcharge          -> float64
total_amount                   -> float64
duration_minutes               -> float64
avg_speed_mph                  -> float64
price_per_mile                 -> float64
pickup_hour                    -> int64
pickup_day                     -> int64
pickup_month    